# Experimento completo em GPU (Kaggle / Colab)

Segmentação do Gray-Scott conforme o `PROTOCOLO.MD` da turma: **U-Net, SegFormer e Mask2Former**, 20 épocas (melhor época pela validação), baseline + P1–P10, baseline + Q1–Q5.

**Kaggle:** *Session options* → Accelerator **GPU T4 x2**, Internet **On** → **Save Version → Save & Run All (Commit)**.

- Com 2 GPUs, o Mask2Former treina numa e U-Net + SegFormer na outra, em paralelo.
- **Se estourar as 12 h:** abra uma nova versão, em *Add Input* anexe a saída da versão anterior (*Notebook Output*) e rode de novo. Os treinos prontos são reaproveitados e só o que falta é treinado.
- No fim, baixe `resultados.zip` na aba **Output** e descompacte na raiz do projeto local.

In [ ]:
REPO = 'https://github.com/JoaoLago10/segmentacao-gray-scott.git'
!git clone -q $REPO projeto || (cd projeto && git pull -q)
%cd projeto
!git log --oneline -1
!nvidia-smi -L

In [ ]:
!pip install -q h5py fsspec aiohttp PyWavelets scikit-image transformers
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.device_count(), 'GPU(s)')

In [ ]:
# Retomada: se a saída de uma versão anterior foi anexada como input, reaproveita treinos e probabilidades.
import glob, shutil, os
prev = [p for p in glob.glob('/kaggle/input/**/results/exp/runs', recursive=True)]
if prev:
    src = os.path.dirname(prev[0])
    print('reaproveitando', src)
    shutil.copytree(src, 'results/exp', dirs_exist_ok=True)
    print(len(os.listdir('results/exp/runs')), 'treinos já prontos')
else:
    print('execução nova (nenhuma saída anterior anexada)')

In [ ]:
!python scripts/04_preparar_dados.py | tail -3   # ~2,8 GB, leitura parcial do Hugging Face

In [ ]:
# Mede a velocidade real nesta GPU e estima o tempo total por arquitetura (~3 min).
!python scripts/05_treinar.py --bench --amp

In [ ]:
# Treino: 2 GPUs em paralelo (ou sequencial com 1 GPU). Logs em log_gpu*.txt.
import subprocess, time, os, re
ngpu = torch.cuda.device_count()
jobs = ([('0', 'mask2former'), ('1', 'unet segformer')] if ngpu >= 2 else [('0', 'unet segformer mask2former')])
procs = []
for gpu, archs in jobs:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
    log = open(f'log_gpu{gpu}.txt', 'a')
    procs.append(subprocess.Popen(f'python scripts/05_treinar.py --arch {archs} --pre all --amp',
                                  shell=True, env=env, stdout=log, stderr=subprocess.STDOUT))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(600)
    for gpu, _ in jobs:
        lines = open(f'log_gpu{gpu}.txt').read().splitlines()
        done = sum(' ok:' in l for l in lines)
        print(f'[{(time.time() - t0) / 3600:.1f} h] GPU {gpu}: {done} treinos prontos | {lines[-1] if lines else ""}', flush=True)
for gpu, _ in jobs:
    print(f'--- GPU {gpu} (fim do log)'); print('\n'.join(open(f'log_gpu{gpu}.txt').read().splitlines()[-5:]))
print('códigos de saída:', [p.returncode for p in procs])

In [ ]:
!MPLCONFIGDIR=.cache/mpl python scripts/06_avaliar.py --workers 4
!MPLCONFIGDIR=.cache/mpl python scripts/07_figuras.py
!python scripts/08_tabelas_latex.py

In [ ]:
!cp log_gpu*.txt results/exp/ 2>/dev/null; zip -qr resultados.zip results/exp docs/artigo/figs/exp docs/artigo/tabelas/exp -x 'results/exp/probs/*' 'results/exp/checkpoints/*'
!ls -lh resultados.zip
!rm -rf data/processed   # os dados são baixados de novo se precisar; probabilidades e checkpoints ficam para retomada